# 06: Brownian-driver to OU-response operator

**Status, 2 September:** acceptance and twelve primary MSE/$J_2$ fits are complete. Signature audit passes; six uniform signature fits are next. Detailed procedure: `../docs/neural_ode_operator_experiments.md` §4.

## Experiment

Experiment B learns a simultaneous stream transformation. Input is one Brownian driver over $[0,1]$; target is OU response driven by same increments over same interval. Model sees complete driver rather than a prefix, and produces complete response rather than forecast. Causal architecture prevents future driver values from changing earlier outputs.

Run order:

```bash
python scripts/run_ou_operator.py --config configs/neural_cde_brownian_ou.yaml --out results/runs/neural_cde_brownian_ou/acceptance --acceptance
python scripts/run_ou_operator_study.py --config configs/neural_cde_brownian_ou.yaml --out results/runs/neural_cde_brownian_ou --stage primary --evaluate-test
python scripts/run_ou_operator.py --config configs/neural_cde_brownian_ou.yaml --out results/runs/neural_cde_brownian_ou/signature_audit --signature-audit
```

Primary stage starts only after acceptance file reports `passed: true`. Signature stage starts only after primary fits and scaling audit are reviewed.


## 1. Paired data

Let $M=256$, $T=1$, $\Delta t=T/M$ and $\tau_m=m\Delta t$. For path $i$, draw independent $\xi_m^{(i)}\sim\mathcal N(0,1)$ and set

$$
\Delta W_m^{(i)}=\sqrt{\Delta t}\,\xi_m^{(i)},\qquad W_0^{(i)}=0,\qquad W_{m+1}^{(i)}=W_m^{(i)}+\Delta W_m^{(i)}.
$$

With $\lambda=2$, $\sigma=0.5$ and $Y_0^{(i)}=0$, target recurrence is

$$
Y_{m+1}^{(i)}=Y_m^{(i)}-\lambda Y_m^{(i)}\Delta t+\sigma\Delta W_m^{(i)}.
$$

Same $\Delta W_m^{(i)}$ appears in driver and response, defining paired operator example $(W^{(i)},Y^{(i)})$. Fixed split seeds generate 512 training, 128 validation and 256 test pairs. Test paths remain unused until configuration and acceptance gates are fixed.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from pathloss.datasets import brownian_ou_pairs

example = brownian_ou_pairs(1, n_steps=256, rng=12001)
figure, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(example['time'], example['driver'][0, :, 0])
axes[0].set_title('Brownian input')
axes[1].plot(example['time'], example['target'][0, :, 0])
axes[1].set_title('Paired OU target')
for axis in axes:
    axis.set_xlabel('time')
figure.tight_layout()


## 2. Target observations

Model receives all 257 driver values. Loss sees $C=64$ response values. Uniform condition uses $t_r=r/(C-1)$. Clustered condition uses

$$
t_r=1-\left(1-\frac{r}{C-1}\right)^3.
$$

Generated response is piecewise-linearly interpolated to these times. Prediction uses same interpolation rule. Driver paths, response paths, model initialization, batch order and optimizer remain shared across paired loss runs. Only target time condition and training discrepancy vary.


## 3. Path-output Neural CDE

Control is $X^{(i)}(t)=(t,W^{(i)}(t))$. Hidden state $h^{(i)}(t)\in\mathbb R^{16}$ starts from shared learned vector $\eta$ and follows

$$
dh^{(i)}(t)=V_{\theta,0}(h^{(i)}(t))\,dt+V_{\theta,1}(h^{(i)}(t))\,dW^{(i)}(t).
$$

$V_\theta:\mathbb R^{16}\to\mathbb R^{16\times2}$ is a two-layer tanh network of width 64. Affine decoder produces $\widehat Y^{(i)}(t)=q_\phi(h^{(i)}(t))$. Decoder acts throughout hidden trajectory. Earlier output therefore depends only on driver prefix.

This differs from final-summary query model in notebook 03. That model reads complete input into one vector before answering every time and cannot represent required causal stream transformation.


## 4. Discrete CDE solver

On control interval $[\tau_m,\tau_{m+1}]$, define increment $\Delta X_m^{(i)}=(\Delta t,\Delta W_m^{(i)})$. Reparameterize interval by $s\in[0,1]$. For hidden value $z$, increment field is

$$
F_m^{(i)}(z)=V_{\theta,0}(z)\Delta t+V_{\theta,1}(z)\Delta W_m^{(i)}.
$$

One RK4 step gives

$$
k_1=F_m^{(i)}(h_m),\quad k_2=F_m^{(i)}(h_m+k_1/2),\quad k_3=F_m^{(i)}(h_m+k_2/2),\quad k_4=F_m^{(i)}(h_m+k_3),
$$

$$
h_{m+1}=h_m+\frac{k_1+2k_2+2k_3+k_4}{6}.
$$

All operations are differentiable. A perturbation beginning after $\tau_m$ first changes $h_{m+1}$, leaving decoded prefix through $\tau_m$ unchanged.


## 5. Losses and training

For response residual $e_r^{(i)}=\widehat Y^{(i)}(t_r)-Y^{(i)}(t_r)$, primary losses are

$$
D_{\mathrm{MSE}}=\frac1C\sum_{r=0}^{C-1}|e_r^{(i)}|^2,\qquad D_{J_2}=\frac1T\sum_{r=0}^{C-1}w_r|e_r^{(i)}|^2,
$$

where $w_r$ are trapezoid elapsed-time weights. Batch loss averages path losses over batch. Three paired seeds use batch size 64, Adam learning rate $10^{-3}$ and 500 epochs: eight batches per epoch and 4,000 parameter updates. Deterministic epoch permutations are shared across losses. Fixed-budget final model is saved; validation does not select an epoch.

Optional signature losses use time-augmented response $(t,Y)$, initial-value anchor and training-target standard deviation as output scale. Global depth four and five local depth-two blocks each store 30 positive-level coordinates. Scaling audit precedes signature training.


## 6. Acceptance before comparison

Primary array runs only after one report confirms:

1. generated responses satisfy stated recurrence;
2. output shape is $(B,C,1)$ and finite gradient reaches every parameter;
3. driver perturbation changes future prediction;
4. perturbing driver after $\tau$ leaves prediction through $\tau$ unchanged within $10^{-7}$;
5. small-batch training loss falls below 10% of first-epoch value and below $10^{-3}$;
6. held-out MSE falls below 80% of untrained value;
7. split paths, target times, initialization and batch order reproduce under same seeds.

Thresholds are implementation gates fixed in configuration before acceptance run. Failure repairs implementation or increases training budget; thresholds remain fixed.


## 7. Evaluation

Every final model is decoded at all 257 fine-grid times. Report fine-grid MSE, $J_1$, $J_2$, $J_4$, $L^\infty$, global and local signature discrepancies, runtime and peak memory. Signature discrepancies are also calculated on the 64-point training polygon. This separates optimization of the specified training objective from agreement on the finer path. Independent dynamics residual is

$$
R_m^{(i)}=\widehat Y_{m+1}^{(i)}-\widehat Y_m^{(i)}+\lambda\widehat Y_m^{(i)}\Delta t-\sigma\Delta W_m^{(i)},
$$

with score $E_{\mathrm{dyn}}=(n_{\mathrm{test}}M)^{-1}\sum_{i,m}|R_m^{(i)}|^2$. No training loss directly optimizes this score.

For condition $q$ and seed $s$, define paired effect

$$
\delta_{q,s}=E_{q,s}^{J_2\text{-trained}}-E_{q,s}^{\mathrm{MSE\text{-trained}}},\qquad \Delta_s=\delta_{\mathrm{clustered},s}-\delta_{\mathrm{uniform},s}.
$$

Elapsed-time mechanism predicts $\Delta_s<0$. Near-zero error for all fits means OU served its intended exact-learnability calibration role.


In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

RESULT_ROOT = REPO_ROOT / 'results/runs/neural_cde_brownian_ou'
rows = []
for path in RESULT_ROOT.glob('seed*/*/*/meta.json'):
    meta = json.loads(path.read_text())
    metrics = meta.get('test') or meta.get('validation')
    if metrics:
        rows.append({
            'seed': meta['fit_config']['seed'],
            'condition': meta['fit_config']['condition'],
            'loss': meta['fit_config']['loss'],
            'initial_fingerprint': meta['initial_fingerprint'],
            'data_fingerprint': meta['data_fingerprint'],
            'target_time_fingerprint': meta['target_time_fingerprint'],
            'order_fingerprint': meta['order_fingerprint'],
            **metrics,
        })
results = pd.DataFrame(rows)
display(results if not results.empty else Markdown('Comparative results are absent.'))


In [ ]:
if not results.empty and {'mse', 'j2'} <= set(results['loss']):
    primary = results[results['loss'].isin(['mse', 'j2'])]
    paired = primary.pivot(index=['seed', 'condition'], columns='loss', values='fine_j2')
    paired['delta'] = paired['j2'] - paired['mse']
    contrast = paired['delta'].unstack('condition')
    contrast['Delta'] = contrast['clustered'] - contrast['uniform']
    display(paired)
    display(contrast)


## 8. Primary result

All twelve primary fits are complete. Uniform observations give no systematic MSE versus $J_2$ difference: across-seed mean fine-grid $J_2$ is $3.67\times10^{-7}$ under both training losses, and the lower error splits across seeds. Under clustered observations, $J_2$ training gives lower fine-grid $J_2$ in all three paired seeds. Ratio of across-seed means is $0.483$, corresponding to a $51.7\%$ reduction relative to MSE training. Predefined sampling contrast $\Delta_s$ is negative for all three seeds.

Clustered $J_2$ training also lowers mean dynamics residual and both dense signature discrepancies. Uniform values remain essentially tied. This transfers the sampling-measure mechanism from fitting one path to learning a causal map across Brownian drivers and held-out OU responses.

Every fit has small absolute error relative to training-target standard deviation $0.213$. Late validation histories fluctuate near this low-error floor; best validation checkpoint does not favour $J_2$ in every clustered seed. Result therefore concerns equal 500-epoch training budgets rather than different attainable optima. OU has served its intended exact-learnability calibration role.

Signature audit reports finite nonzero component values and parameter gradients at every retained level. Training-target standard-deviation scaling is frozen. Higher-level gradients are smaller than anchor and level-one gradients, so acceptance authorizes comparative runs without asserting balanced or easy optimization.
